#################################################
# End-to-End AWS Text Classification System 
# With Multiple Models + Voting 
#################################################

In [1]:
# !pip install xgboost
import os
import pandas as pd
import numpy as np
import html
import unicodedata
import re
import string
import nltk
import joblib
import mlflow
import mlflow.sklearn
import time

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer

# XGBoost
from xgboost import XGBClassifier

# Download NLTK dependencies
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\youss\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\youss\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\youss\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


#############################
### 1. Read Data
#############################

In [2]:
data_reading_time =time.time()
df = pd.read_csv("D:/1)debi/project/IMDB Dataset.csv")  # 50K IMDB reviews dataset
end_data_reading_time = time.time()
Reading_Time = end_data_reading_time - data_reading_time
print("Reading Data Time :", Reading_Time, "Sec")
print("Data Sample:")
display(df.head())

Reading Data Time : 1.2441484928131104 Sec
Data Sample:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


#############################
### 2. Text Preprocessing
#############################

In [3]:
stop_words = set(stopwords.words('english'))

def remove_special_chars(text):
    re1 = re.compile(r'  +')
    x1 = text.lower().replace('#39;', "'").replace('amp;', '&').replace('#146;', "'") \
            .replace('nbsp;', ' ').replace('#36;', '$').replace('\\n', "\n") \
            .replace('quot;', "'").replace('<br />', "\n").replace('\\"', '"') \
            .replace('<unk>', 'u_n').replace(' @.@ ', '.').replace(' @-@ ', '-') \
            .replace('\\', ' \\ ')
    return re1.sub(' ', html.unescape(x1))

def remove_non_ascii(text):
    return unicodedata.normalize('NFKD', text).encode('ascii','ignore').decode('utf-8','ignore')

def to_lowercase(text):
    return text.lower()

def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

def replace_numbers(text):
    return re.sub(r'\d+', '', text)

def text2words(text):
    return word_tokenize(text)

def remove_stopwords(words):
    return [word for word in words if word not in stop_words]

def lemmatize_words(words):
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(word) for word in words]

def normalize_text(text):
    """Applies all preprocessing steps to text"""
    text = remove_special_chars(text)
    text = remove_non_ascii(text)
    text = remove_punctuation(text)
    text = to_lowercase(text)
    text = replace_numbers(text)
    words = text2words(text)
    words = remove_stopwords(words)
    words = lemmatize_words(words)
    return ' '.join(words)

In [4]:
Text_Preprocessing_time =time.time()
# Apply the complete pipeline
df['cleaned_review'] = df['review'].apply(normalize_text)
End_Text_Preprocessing_time=time.time()
Preprocessing_time= End_Text_Preprocessing_time - Text_Preprocessing_time
print("Text Preprocessing Time :", Preprocessing_time, "Sec")

print("\nCleaned Data Sample:")
display(df[['review','cleaned_review']].head())

Text Preprocessing Time : 61.993687868118286 Sec

Cleaned Data Sample:


,review,cleaned_review
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


#############################
### 3. Feature Extraction (TF-IDF)
#############################

In [5]:
# Convert sentiment to numeric (0 = negative, 1 = positive)
df_one_hot = pd.get_dummies(df, columns=['sentiment'], dtype=int)
y = df_one_hot['sentiment_positive'].values

# Split text first (not vectorized)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['cleaned_review'], y, test_size=0.2, random_state=42)

# Initialize and fit TF-IDF on training text only
tfidf = TfidfVectorizer(max_features=10000)
start_tfidf =time.time()
X_train = tfidf.fit_transform(X_train_text)
end_tfidf=time.time()
tfidf_time= end_tfidf - start_tfidf
print("The TF_IDF fitting time is: ", tfidf_time, "Sec")

X_test = tfidf.transform(X_test_text)

joblib.dump(tfidf, "models/tfidf.pkl")

print("\nShapes after TF-IDF (fit only on training data):")
print("Training set:", X_train.shape, "Test set:", X_test.shape)

The TF_IDF fitting time is:  4.285511493682861 Sec

Shapes after TF-IDF (fit only on training data):
Training set: (40000, 10000) Test set: (10000, 10000)


#############################
### 4. Training & Evaluation Helper
#############################

In [6]:
def train_and_evaluate(model, name, X_train, y_train, X_test, y_test, path="models/"):
    
    # Train
    print(f"Training: {name}")
    start_train = time.time()
    model.fit(X_train, y_train)
    end_train = time.time()
    train_time = end_train - start_train
    print("Training Time: ",train_time, "Sec")

    # Test
    start_test = time.time()
    preds = model.predict(X_test)
    end_test =time.time()
    test_time =  end_test- start_test
    print("Testing Time: ",test_time, "Sec")
    
    # Accuracy
    acc = accuracy_score(y_test, preds)
    report = classification_report(y_test, preds)
    print(f"[{name}] Accuracy: {acc:.4f}")
    print(report)

    # Save model
    filepath = f"{path}{name}.pkl"
    joblib.dump(model, filepath)

    # MLFLOW
    mlflow.log_param("model", name)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("training_time", train_time)
    mlflow.log_metric("testing_time", test_time)
    mlflow.sklearn.log_model(model, name)

    return model, preds, acc, filepath

In [ ]:
def load_trained_models(model_names, path="models/"):
    loaded = {}
    for name in model_names:
        model_path = os.path.join(path, f"{name}.pkl")
        if os.path.exists(model_path):
            loaded[name] = joblib.load(model_path)
        else:
            print(f"Warning: {model_path} not found.")
    return loaded


#############################
### 5. Define Multiple Models 
#############################

In [8]:

log_reg = LogisticRegression(max_iter=1000, random_state=42)
naive_bayes = MultinomialNB()
rf = RandomForestClassifier(n_estimators=100, random_state=42)
svm = SVC(kernel='linear', probability=True, random_state=42)
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

models = {
    "LogisticRegression": log_reg,
    "NaiveBayes": naive_bayes,
    "RandomForest": rf,
    "SVM": svm,
    "XGBoost": xgb  
}

trained_models = {}

### Train Models

In [9]:

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        fitted_model, _, _, _ = train_and_evaluate(model, name, X_train, y_train, X_test, y_test)
        trained_models[name] = fitted_model

Training: LogisticRegression
Training Time:  0.5061986446380615
Testing Time:  0.0
[LogisticRegression] Accuracy: 0.8926
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.88      0.91      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



2025/04/12 21:29:16 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Training: NaiveBayes
Training Time:  0.015387773513793945
Testing Time:  0.008856058120727539
[NaiveBayes] Accuracy: 0.8560
              precision    recall  f1-score   support

           0       0.85      0.86      0.85      4961
           1       0.86      0.86      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000



2025/04/12 21:29:20 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Training: RandomForest
Training Time:  118.75783371925354
Testing Time:  0.5086498260498047
[RandomForest] Accuracy: 0.8537
              precision    recall  f1-score   support

           0       0.84      0.87      0.85      4961
           1       0.86      0.84      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



...

2025/04/12 22:55:31 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Training: XGBoost


c:\Users\youss\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:158: UserWarning: [22:55:32] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training Time:  79.77446031570435
Testing Time:  0.03397679328918457
[XGBoost] Accuracy: 0.8605
              precision    recall  f1-score   support

           0       0.87      0.84      0.86      4961
           1       0.85      0.88      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000



2025/04/12 22:56:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


### Load Models

In [ ]:
model_names = ["LogisticRegression", "NaiveBayes", "RandomForest", "SVM", "XGBoost"]
trained_models = load_trained_models(model_names)

### View Loaded Models

In [12]:
for name, model in trained_models.items():
    print(f"Model: {name}")
    print(model)
    print("-" * 40)


Model: LogisticRegression
LogisticRegression(max_iter=1000, random_state=42)
----------------------------------------
Model: NaiveBayes
MultinomialNB()
----------------------------------------
Model: RandomForest
RandomForestClassifier(random_state=42)
----------------------------------------
Model: SVM
SVC(kernel='linear', probability=True, random_state=42)
----------------------------------------
Model: XGBoost
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_we

models = {
    "LogisticRegression": log_reg,
    "NaiveBayes": naive_bayes,
    "RandomForest": rf,
    "SVM": svm,
    "XGBoost": xgb  
}

In [ ]:
from scipy.stats import mode

CLASSES_LIST = ['negative', 'positive']

# Input review
x = input("Enter a review: ")
print("The entered Text is: ", x)
# Clean and vectorize
x_clean = normalize_text(x)
print("cleaned input text: ",x_clean)
x_vector = tfidf.transform([x_clean])

# Collect predictions
predictions = []

for name, model in trained_models.items():
    pred = model.predict(x_vector)[0]
    predictions.append(pred)
    print(f"{name} predicted: {CLASSES_LIST[pred]}")

# Get most common prediction
final_pred = mode(predictions, keepdims=False).mode
print(f"\nFinal Prediction (by majority vote): {CLASSES_LIST[final_pred]}")


LogisticRegression predicted: negative
NaiveBayes predicted: negative
RandomForest predicted: negative
SVM predicted: negative
XGBoost predicted: negative

Final Prediction (by majority vote): negative


In [ ]:
voting_clf = VotingClassifier(
    estimators=[(name, model) for name, model in trained_models.items()],
    voting='hard'
)
voting_clf = joblib.load("models/VotingClassifier.pkl")
CLASSES_LIST = ['negative', 'positive']
x = input("Enter a review: ")
print("The entered Text is: ", x)
x_clean = normalize_text(x)
print("cleaned input text: ",x_clean)
# Use the same TF-IDF vectorizer used in training
tfidf = joblib.load("models/tfidf.pkl")
x_vector = tfidf.transform([x_clean])

# Access individual estimators
for name, model in voting_clf.named_estimators_.items():
    pred = model.predict(x_vector)[0]
    print(f"{name} predicted: {CLASSES_LIST[pred]}")

# Final ensemble prediction
final_pred = voting_clf.predict(x_vector)[0]
print(f"\nFinal Prediction (Voting Classifier): {CLASSES_LIST[final_pred]}")



LogisticRegression predicted: negative
NaiveBayes predicted: negative
RandomForest predicted: negative
SVM predicted: negative
XGBoost predicted: negative

Final Prediction (Voting Classifier): negative


In [10]:

voting_clf = VotingClassifier(
    estimators=[(name, model) for name, model in trained_models.items()],
    voting='hard'
)

voting_clf.fit(X_train, y_train)
 
joblib.dump(voting_clf, "models/VotingClassifier.pkl")

# Evaluate ensemble
ensemble_preds = voting_clf.predict(X_test)
acc = accuracy_score(y_test, ensemble_preds)
print(f"\nVoting Classifier Accuracy: {acc:.4f}")
print(classification_report(y_test, ensemble_preds))

c:\Users\youss\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:158: UserWarning: [00:14:10] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Voting Classifier Accuracy: 0.8916
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.89      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [11]:
#############################
### 8. Testing on a Small Sample of Test Data
#############################

print("\n--- Sample Test Predictions on the Test Set ---")
sample_indices = [0, 1, 2, 3, 4]  # first 5 test examples, for instance
for idx in sample_indices:
    review_vector = X_test[idx]
    true_label = y_test[idx]
    ensemble_prediction = voting_clf.predict(review_vector)[0]
    # Map numeric to label
    pred_label_str = "POSITIVE" if ensemble_prediction == 1 else "NEGATIVE"
    true_label_str = "POSITIVE" if true_label == 1 else "NEGATIVE"
    print(f"Test Sample {idx}: True = {true_label_str}, Predicted = {pred_label_str}")


--- Sample Test Predictions on the Test Set ---
Test Sample 0: True = POSITIVE, Predicted = NEGATIVE
Test Sample 1: True = POSITIVE, Predicted = POSITIVE
Test Sample 2: True = NEGATIVE, Predicted = NEGATIVE
Test Sample 3: True = POSITIVE, Predicted = POSITIVE
Test Sample 4: True = NEGATIVE, Predicted = NEGATIVE


In [12]:
#############################
### 9. Custom Text Prediction
#############################

def predict_custom_text(text, vectorizer, model):
    """
    Preprocess custom text, transform it with the same TF-IDF vectorizer,
    then predict with the provided model. Return predicted sentiment label.
    """
    # Preprocess
    cleaned = normalize_text(text)
    # Vectorize
    transformed = vectorizer.transform([cleaned])
    # Predict
    pred = model.predict(transformed)[0]
    return "POSITIVE" if pred == 1 else "NEGATIVE"

# Allow user to enter custom text (in a real Jupyter environment, you can run this cell and type input)
print("\n--- Custom Text Prediction ---")
user_text = input("Enter a movie review to classify (e.g., 'I loved this movie!'): ")

# Use the voting ensemble as final predictor
prediction_label = predict_custom_text(user_text, tfidf, voting_clf)
print(f"\nYour review is predicted as: {prediction_label}")


--- Custom Text Prediction ---

Your review is predicted as: NEGATIVE


In [ ]:
# MLFLOW UI
import os
os.system("mlflow ui --port 5000 &")
print("MLflow UI started at http://127.0.0.1:5000")